[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Training Loops &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import io
import os
import re

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' tensors, loaders and functions, rebuilt here so that each answer runs on its own.

In [2]:
def as_tensors(rows, target):
    """The standardized features of these rows, and their target as a column."""
    return torch.from_numpy(scaled[rows]), torch.from_numpy(target[rows]).unsqueeze(1)


def train_one_epoch(model, loader, optimizer, loss_fn):
    """One pass over the loader's batches, with a step for each; returns the average loss."""
    model.train()
    total, count = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        count += len(xb)
    return total / count


def evaluate(model, loader, loss_fn):
    """The average loss over the loader's examples, in evaluation mode, with no gradients."""
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            total += loss_fn(model(xb), yb).item() * len(xb)
            count += len(xb)
    return total / count


X_train, y_train = as_tensors(train_rows, jan_mean)
X_valid, y_valid = as_tensors(valid_rows, jan_mean)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(0))
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=200)

**1.** Train a fresh 4-32-1 network from `torch.manual_seed(0)` with `torch.optim.SGD` at a learning rate of 0.05 for 40 epochs, using `train_one_epoch` and `evaluate`, and print the epoch with the lowest validation loss.

In [3]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
optimizer = torch.optim.SGD(net.parameters(), lr=0.05)
mse = torch.nn.MSELoss()
val_losses = []
for epoch in range(40):
    train_one_epoch(net, train_loader, optimizer, mse)
    val_losses.append(evaluate(net, valid_loader, mse))
best = int(np.argmin(val_losses))
print(f"lowest validation loss at epoch {best + 1}: {val_losses[best]:.3f}")

lowest validation loss at epoch 19: 0.765


SGD without momentum, at 0.05, reaches its lowest validation loss, 0.765, at epoch 19, where Adam's in the worked examples was 0.775 at epoch 33. That is one seed, and another seed can reverse both differences.

**2.** Fit the Keras network again from `keras.utils.set_random_seed(0)` with `batch_size=32`, and print the validation loss at epochs 10, 20, 30 and 40.

In [4]:
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
fitted = net_keras.fit(scaled[train_rows], jan_mean[train_rows],
                       validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                       epochs=40, batch_size=32, verbose=0)
for epoch in (10, 20, 30, 40):
    print(f"epoch {epoch}: validation loss {fitted.history['val_loss'][epoch - 1]:.3f}")

epoch 10: validation loss 0.876
epoch 20: validation loss 0.811
epoch 30: validation loss 0.797
epoch 40: validation loss 0.786


With batches of 32 the validation loss reaches 0.786 by epoch 40, against 0.769 with batches of 64 from one seed, after twice as many steps an epoch.

**3.** Find the epoch with the highest validation accuracy in `fitted_snow.history`, with `np.argmax`, and print it with its accuracy.

In [5]:
keras.utils.set_random_seed(0)
classifier_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
classifier_keras.compile(optimizer=keras.optimizers.Adam(0.01),
                         loss=keras.losses.BinaryCrossentropy(from_logits=True),
                         metrics=[keras.metrics.BinaryAccuracy(threshold=0.0)])
fitted_snow = classifier_keras.fit(scaled[train_rows], snow_on_may_1[train_rows],
                                   validation_data=(scaled[valid_rows], snow_on_may_1[valid_rows]),
                                   epochs=40, batch_size=64, verbose=0)

accuracies = fitted_snow.history["val_binary_accuracy"]
best = int(np.argmax(accuracies))
print(f"highest validation accuracy at epoch {best + 1}: {accuracies[best]:.3f}")

highest validation accuracy at epoch 8: 0.890


The validation accuracy is highest at epoch 8, 0.89, against 0.865 at epoch 40. With 200 validation stations, each one is worth 0.005 of accuracy, so the difference is five stations.

**4.** Train a three-band classifier with the PyTorch loop, where the bands are `np.digitize(jan_mean, [-8.0, -3.0])` as in the **Losses and Optimizers** notebook, a 4-32-3 network and `CrossEntropyLoss`, for 40 epochs, and print its validation accuracy.

In [6]:
band = np.digitize(jan_mean, [-8.0, -3.0])
band_loader = DataLoader(TensorDataset(X_train, torch.from_numpy(band[train_rows])), batch_size=64,
                         shuffle=True, generator=torch.Generator().manual_seed(0))

torch.manual_seed(0)
band_net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 3))
optimizer = torch.optim.Adam(band_net.parameters(), lr=0.01)
for epoch in range(40):
    train_one_epoch(band_net, band_loader, optimizer, torch.nn.CrossEntropyLoss())

band_net.eval()
with torch.no_grad():
    predicted = band_net(X_valid).argmax(dim=1)
print(f"validation accuracy: {(predicted == torch.from_numpy(band[valid_rows])).float().mean().item():.3f}")

validation accuracy: 0.860


The three-band network puts 86% of the validation stations in the right band.

---

&#8592; **Back to:** [Training Loops](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/10-training-loops.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
